## actualizar retiro telef 

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [2]:
query = f"""
	select * from Alice.prospectos_correos_alfin
"""
df_correo = pd.read_sql(query, engine_mysql)


query = f"""
	select * from Alice.prospectos_envio_alfin
"""
df_formulario = pd.read_sql(query, engine_mysql)

In [5]:
df_correo[df_correo['fecha_visita']=='2026-07-11'].head()

,id,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,fecha_envio,intentos_realizados,estado,tipo_carga,fecha_registro,fecha_dia


In [17]:
query = f"""
	select NUMERO_DOCUMENTO as dni_cliente,color_final as color from DANTALION.dbo.Base_Maestra_Alfin_bk_Vigente
"""
df_maestra = pd.read_sql(query, engine_kishin)

df_maestra["dni_cliente"] = (
    df_maestra["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_maestra.columns.tolist())


['dni_cliente', 'color']


In [18]:
filename='fomato_agendas_alfin_credicash_2026.xlsx'
filePath = os.path.join(ruta_csv, filename)
df_formato = pd.read_excel(filePath)
df_formato['supervisor']='CARLOS ENRIQUE RAMIREZ CACHIQUE'
df_formato['canal_campo']='CALL CENTER / TARGET OUTSOURCING'
df_formato['codigo_ejecutivo_id']='00000001'
df_formato['ejecutivo_target']='BOT'
df_formato['cdv_alfin_banco']='ROSA HONOR'

df_formato = df_formato.rename(columns={
    'monto': 'monto_solicitado'
})
df_formato["dni_cliente"] = (
    df_formato["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
df_formato = df_formato.merge(
    df_maestra,
    on='dni_cliente',
    how='left'
)

df_formato['fecha_visita']='2026-07-10'

# Semilla opcional para reproducibilidad
# np.random.seed(123)

# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_formato))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_formato))

# Crear la columna
df_formato["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

df_formato['telefono_cliente']=df_formato['celular']
df_formato['dni_vendedor']=df_formato['ejecutivo_target']
df_formato['agencia_tienda']=df_formato['cod_agencia']
df_formato['operador']='TARGET'
df_formato['tipo_gestion']='Derivacion'

In [19]:
equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace(equivalencias)
)
# df_formato.drop_duplicates(subset=["dni"], inplace=True)

In [20]:
df_formato.drop_duplicates(subset=["dni_cliente"], inplace=True)



In [21]:
df_formato.shape

(1135, 19)

In [20]:
# df_formato = df_formato[
#     df_formato['agencia_atencion'].isin([
#         'SAN JUAN DE LURIG',
#         'ENMANCIPACION',
#         'PC HUANCAYO',
#         'TRUJ CENTRO',
#         'PC TACNA',
#         'PC HUARAZ',
#         'TRUJ AMERICA',
#         'AREQ CAYMA',
#         'AREQ PAMPILLA'
#     ])
# ]

#### Validar el nombre de la agencia

In [22]:
query = f"""
	select * from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)

set_correo = set(
    df_formato['agencia_atencion']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_correo']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'ICA'}
set()


In [23]:
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'ICA'}
set()


#### validar el codigo de agencia 

In [24]:

set_correo = set(
    df_formato['agencia_tienda']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_Formulario']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'730879 - PAITA', '739580 - ICA'}
set()


In [26]:
df_correo['tipo_carga']='MANUAL'

In [25]:
df_correo=df_formato[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita']] .copy()

df_formulario=df_formato[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,70286844,FRANK ALEXIS NOLAZCO CUZCANO,NaN,2800,965739962,CAÑETE,2026-07-10,13:45:00
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,71590543,CRISTOPHER EDDIN MANUEL CHUMAN FARROÑAN,NaN,2900,994413738,MOSHOQUEQUE,2026-07-10,18:00:00


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,70286844,FRANK ALEXIS NOLAZCO CUZCANO,965739962,732243 - CAÑETE,2026-07-10,2800,Derivacion
1,BOT,TARGET,71590543,CRISTOPHER EDDIN MANUEL CHUMAN FARROÑAN,994413738,738360 - MOSHOQUEQUE,2026-07-10,2900,Derivacion


In [27]:

df_correo.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_formulario.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

1135

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,09750804,AMADO VILLALTA,NaN,23000,942707381,CASTILLA,2026-07-08,14:45:00
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,08931784,LUIS GUILLERMO CHUQUIJAJAS,NaN,23000,950955962,VILLA EL SALVADOR 2,2026-07-08,14:30:00


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,00000001,TARGET,09750804,AMADO VILLALTA,942707381,737490 - CASTILLA,2026-07-08,23000,Derivacion
1,00000001,TARGET,08931784,LUIS GUILLERMO CHUQUIJAJAS,950955962,732249 - VILLA EL SALVADOR 2,2026-07-08,23000,Derivacion
